# Imports

In [ ]:
import pandas as pd
import psycopg2
import numpy as np
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from psycopg2.extras import execute_batch
df = pd.read_csv('avocado.csv', index_col = 0)
pop = pd.read_csv('POPTHM.csv')

#store save password
with open('pwd.txt', 'w') as file:
    file.write(input('Enter your PostgreSQL password: '))

with open('pwd.txt', 'r') as file:
    pwd = file.read()

## Task 1 (R)

The dataset has three PLU codes (4046, 4225, 4770) representing different avocado sizes. Calculate the average price per PLU type by dividing total value by volume for each. Create separate price series for each size category.

Connect to New Database & Create Table

In [ ]:
usr_nm  = 'postgres'
port    = '5432'
DB_NAME = 'avocado_db'

#creation of the database
conn = psycopg2.connect(
    host='localhost',
    database='postgres',
    user=usr_nm,
    password=pwd,
    port=port
)
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cur = conn.cursor()

try:
    cur.execute(f'CREATE DATABASE {DB_NAME};')
except Exception as e:
    print(f'Note: {e}')
else:
    print('Database created successfully!')
finally:
    cur.close()
    conn.close()

Enter your PostgreSQL password: 


OperationalError: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?


- **Password file** — hardcoding credentials is unsafe. Writing to `pwd.txt` and reading it back at runtime keeps the secret out of the notebook source
- **Connection to `postgres` first** — we cannot connect to `avocado_db` before it exists, so we bootstrap via the default system database
- **`ISOLATION_LEVEL_AUTOCOMMIT`** — `CREATE DATABASE` must run outside any open transaction; autocommit satisfies that constraint
- The `try/except/else/finally` block ensures the cursor and connection are always closed even if the DB already exists

In [ ]:
#connection to 'avocado_db'
conn = psycopg2.connect(
    host='localhost',
    database=DB_NAME,
    user=usr_nm,
    password=pwd,
    port=port
)
cur = conn.cursor()
#colums creation
cur.execute("""
    DROP TABLE IF EXISTS avocado_data;
    CREATE TABLE avocado_data (
        id              SERIAL PRIMARY KEY,
        "Date"          DATE            NOT NULL,
        "AveragePrice"  NUMERIC(10, 4)  NOT NULL,
        "Total Volume"  NUMERIC(20, 2)  NOT NULL,
        "4046"          NUMERIC(20, 2),
        "4225"          NUMERIC(20, 2),
        "4770"          NUMERIC(20, 2),
        "Total Bags"    NUMERIC(20, 2),
        "Small Bags"    NUMERIC(20, 2),
        "Large Bags"    NUMERIC(20, 2),
        "XLarge Bags"   NUMERIC(20, 2),
        "type"          VARCHAR(20)     NOT NULL,
        "year"          INTEGER         NOT NULL,
        "region"        VARCHAR(100)    NOT NULL
    );
""")

conn.commit()
cur.close()
conn.close()
print('Table created successfully!')

- **`DROP TABLE IF EXISTS avocado_data`** — drops the table
- **`CREATE TABLE`** — creates the table that will be filled with data from dataset
- **`SERIAL PRIMARY KEY`** — auto-incrementing surrogate key enforces row uniqueness
- **`NUMERIC(precision, scale)`** — exact decimal arithmetic avoids floating-point rounding errors in percentage calculations

Insert Data

In [ ]:
df = pd.read_csv('avocado.csv')

conn = psycopg2.connect(
    host='localhost',
    database=DB_NAME,
    user=usr_nm,
    password=pwd,
    port=port
)
cur = conn.cursor()

#build a list of tuples, one tuple per CSV row, to feed into execute_batch
#df.iterrows() yields (index, row) pairs for every row in the DataFrame.
#we use implicit types because psycopg2 is not very good with pandas data types

rows = [
    (
        str(row['Date']),
        float(row['AveragePrice']),
        float(row['Total Volume']),
        float(row['4046']),
        float(row['4225']),
        float(row['4770']),
        float(row['Total Bags']),
        float(row['Small Bags']),
        float(row['Large Bags']),
        float(row['XLarge Bags']),
        str(row['type']),
        int(row['year']),
        str(row['region'])
    )

    for _, row in df.iterrows()#a different way to make a loop with data without appends
]
# Send all rows to the database in batches using execute_batch
execute_batch(
    cur,
    """
    INSERT INTO avocado_data
        ("Date", "AveragePrice", "Total Volume",
         "4046", "4225", "4770",
         "Total Bags", "Small Bags", "Large Bags", "XLarge Bags",
         "type", "year", "region")
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    """,
    rows,
    page_size=500
)
#apply the changes
conn.commit()
#print number of rows that has been imported to make sure everything is ok
cur.execute('SELECT COUNT(*) FROM avocado_data;')
data    = cur.fetchall()#method that retrieves every single row returned by SELECT statement
columns = [desc[0] for desc in cur.description]
print(pd.DataFrame(data=data, columns=columns))

cur.close()
conn.close()

- **List-of-tuples comprehension** — converts each value to a native Python type (`float`/`int`/`str`) to avoid NumPy serialisation(that Pandas uses) issues with psycopg2
- **`execute_batch` with `page_size=500`** — groups rows into batches of 500, drastically reducing network round-trips

PLU Price Series

In [ ]:
conn = psycopg2.connect(
    host='localhost', database=DB_NAME,
    user=usr_nm, password=pwd, port=port
)
cur = conn.cursor()

cur.execute("""
    SELECT
        "Date",
        "region",
        "type",

        -- взвешенная средняя цена для PLU 4046 (small Hass):
        -- каждая неделя вносит вклад пропорционально своему объёму
        -- SUM(price * vol_4046) / SUM(vol_4046)
        ROUND(
            SUM("AveragePrice" * "4046") / NULLIF(SUM("4046"), 0)
        , 4) AS wavg_price_4046,

        -- PLU 4225 (large Hass)
        ROUND(
            SUM("AveragePrice" * "4225") / NULLIF(SUM("4225"), 0)
        , 4) AS wavg_price_4225,

        -- PLU 4770 (extra-large Hass)
        ROUND(
            SUM("AveragePrice" * "4770") / NULLIF(SUM("4770"), 0)
        , 4) AS wavg_price_4770

    FROM avocado_data
    WHERE ("4046" + "4225" + "4770") > 0
    GROUP BY "Date", "region", "type"
    ORDER BY "region", "type", "Date";
""")

data    = cur.fetchall()
columns = [desc[0] for desc in cur.description]
t1      = pd.DataFrame(data=data, columns=columns)
print(t1.head(10).to_string())
cur.close()
conn.close()

## Task 2 (S)

Calculate per capita consumption by dividing total volume by regional population. Create market penetration metrics: share of households buying avocados (proxy using volume relative to regional averages).

In [ ]:
def connect(pwd: str,
            usr_nm: str = 'postgres',
            host: str = 'localhost',
            port: str = '5432',
            db_nm: str = 'postgres'
           ) -> psycopg2.extensions.connection | None:

    conn = None

    try:
        conn = psycopg2.connect(
            host=host,
            database=db_nm,
            user=usr_nm,
            port=port,
            password=pwd
        )

    except:
        print('Connection failure!')

    else:
        print('Connection success!')

    finally:
        return conn


def create_db(new_db_nm: str,
              conn: psycopg2.extensions.connection | None = None,
              cur: psycopg2.extensions.cursor | None = None
             ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:

    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"

    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False

    if conn is not None:
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

        cur = conn.cursor()

    try:
        cur.execute(f'CREATE DATABASE {new_db_nm};')

    except:
        print(f"Couldn't create database. The database {new_db_nm} probably already exists")

    else:
        print(f'Database {new_db_nm} created!')

    finally:
        if conn_flg:
            return conn, cur
        else:
            return cur


def create_table(table_nm: str,
                 table_cols: list[str],
                 table_dtypes: list[str],
                 conn: psycopg2.extensions.connection | None = None,
                 cur: psycopg2.extensions.cursor | None = None
                ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:

    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"

    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False

    if conn is not None:
        cur = conn.cursor()


    pairs = []

    for col, dtype in zip(table_cols, table_dtypes):
        pairs.append(f'"{col}" {dtype},') # pairs.append(f'"{col}" {dtype},') pairs.append(' '.join([col, dtype])+',')

    pairs[-1] = pairs[-1].strip(',')

    cols = '\n'.join(pairs)

    try:
        query = f'''CREATE TABLE {table_nm} (
                id SERIAL,
                {cols}
                    );'''

        cur.execute(query)

        conn.commit()

    except Exception as e:
        print('Failed to create table, such table already exists')
        print(str(e))

    else:
        print(f'Succesfully created table {table_nm}')

    finally:

        if conn_flg:
            return conn, cur
        else:
            return cur



def upload_to_table(table_nm: str,
                    df: pd.DataFrame,
                    conn: psycopg2.extensions.connection | None = None,
                    cur: psycopg2.extensions.cursor | None = None
                   ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor] | psycopg2.extensions.cursor:

    assert conn is not None or cur is not None, "At least one of the following variables should be passed: conn, cur"

    conn_flg = True if conn is not None else False
    cur_flg = True if cur is not None else False

    if conn is not None:
        cur = conn.cursor()

    try:

        for idx, row_data in df.iterrows():
            cur.execute(f'INSERT INTO {table_nm} ({', '.join(f'"{col}"' for col in df.columns)}) VALUES ({('%s,'*len(df.columns)).strip(',')})', tuple(row_data))

    except Exception as e:
        print('Failed to append data to SQL table!')
        print(str(e))

    else:
        print(f'Added data to table {table_nm}')

    finally:
        if conn_flg:
            return conn, cur
        else:
            return cur

Cpc - consumption per capita | c-type - conventional, o-type - organic


,Date,US avocado(c) Cpc,US avocado(o) Cpc
0,2015-12-27,0.084431,0.001700
1,2015-12-20,0.077582,0.001644
2,2015-12-13,0.086730,0.001931
3,2015-12-06,0.089078,0.001590
4,2015-11-29,0.070000,0.001572
...,...,...,...
7,2018-02-04,0.190517,0.004221
8,2018-01-28,0.122486,0.004077
9,2018-01-21,0.130926,0.003915
10,2018-01-14,0.113730,0.004502


Mean Cpc for conventional avocado across the US is therefore approximately 0.104 units.
Mean Cpc for organic avocado across the US is therefore approximately 0.003 units.


This creates functions to operate with preSQL easily in the main body. Almost the same as in classes, but I have added comas to cols name in table creation function, since it didn't work without that change.

In [ ]:
def main():

    conn = connect(pwd)
    conn, cur = create_db('avocado', conn=conn)
    conn.close()

    conn = connect(pwd, db_nm='avocado')
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS avocado_table")
    cur.execute("DROP TABLE IF EXISTS population")
    conn.commit()

    types_mapping = {
        'int64': 'INTEGER',
        'float64': 'NUMERIC',
        'object': 'VARCHAR(100)',
    }
    avocado_dtypes = [types_mapping[str(dt)] for dt in df.dtypes]
    pop_dtypes = [types_mapping[str(dt)] for dt in pop.dtypes]

    create_table('avocado_table', df.columns.tolist(), avocado_dtypes, conn=conn, cur=cur)
    create_table('population', pop.columns.tolist(), pop_dtypes, conn=conn, cur=cur)

    upload_to_table('avocado_table', df, conn=conn, cur=cur)
    upload_to_table('population', pop, conn=conn, cur=cur)

    cur.execute("""
      WITH avocado_month AS (
        SELECT "Date", "type", "Total Volume", TO_CHAR(TO_DATE("Date", 'YYYY-MM-DD'), 'YYYY-MM') AS year_month
        FROM avocado_table
        WHERE region = 'TotalUS'
      ),
      pop_month AS (
        SELECT TO_CHAR(TO_DATE("observation_date", 'YYYY-MM-DD'), 'YYYY-MM') AS year_month, "POPTHM" * 1000 AS pop
        FROM population
      )
      SELECT "Date", "type", ROUND("Total Volume" / pop, 3) AS cpc
      FROM avocado_month
      JOIN pop_month ON avocado_month.year_month = pop_month.year_month
      ORDER BY "Date", "type";
      """)

    final = pd.DataFrame(cur.fetchall(), columns=['Date', 'type', 'US_total_avocado_cpc'])
    print(final)



main()

Add the population metric date-by-date and dividing Total Volume by it to get the required metric.

## Task 3 (I)

Calculate short-term price elasticity for each region: % change in volume / % change in price from week to week. Smooth this with 4-week rolling average to reduce noise. Handle extreme values (elasticity > 5 or < -5).

In [ ]:
def regional_short_elasticity(i):
    df_c: pd.DataFrame = reg[(reg['region'] == i) & (reg['type'] == 'conventional')].copy()
    df_o: pd.DataFrame = reg[(reg['region'] == i) & (reg['type'] == 'organic')].copy()

    df_c = df_c.sort_values('Date').reset_index(drop=True)
    df_o = df_o.sort_values('Date').reset_index(drop=True)

    if df_c.empty or df_o.empty:
        t4: pd.DataFrame = pd.DataFrame({
            'region': [i, i],
            'type': ['conventional', 'organic'],
            'elasticity': [np.nan, np.nan]
        })
        return t4, np.array([]), np.array([])

    dlog_p_c = df_c['log_price'].diff().values
    dlog_v_c = df_c['log_volume'].diff().values
    dlog_p_o = df_o['log_price'].diff().values
    dlog_v_o = df_o['log_volume'].diff().values

    eps: float = 1e-8
    mask_c = np.abs(dlog_p_c) > eps
    mask_o = np.abs(dlog_p_o) > eps

    el_c_raw: np.ndarray = np.full_like(dlog_p_c, np.nan)
    el_o_raw: np.ndarray = np.full_like(dlog_p_o, np.nan)

    el_c_raw[mask_c] = dlog_v_c[mask_c] / dlog_p_c[mask_c]
    el_o_raw[mask_o] = dlog_v_o[mask_o] / dlog_p_o[mask_o]

    el_c_clipped: np.ndarray = np.clip(el_c_raw, -5, 5)
    el_o_clipped: np.ndarray = np.clip(el_o_raw, -5, 5)


    ser_c = pd.Series(el_c_clipped, index=df_c.index)
    ser_o = pd.Series(el_o_clipped, index=df_o.index)

    el_c_smooth = ser_c.rolling(window=4, min_periods=4).mean().values
    el_o_smooth = ser_o.rolling(window=4, min_periods=4).mean().values

    mean_c = np.nanmean(el_c_smooth) if np.any(~np.isnan(el_c_smooth)) else np.nan
    mean_o = np.nanmean(el_o_smooth) if np.any(~np.isnan(el_o_smooth)) else np.nan

    III = pd.DataFrame({
        'region': [i, i],
        'type': ['conventional', 'organic'],
        'elasticity': [
            round(mean_c, 3) if not np.isnan(mean_c) else np.nan,
            round(mean_o, 3) if not np.isnan(mean_o) else np.nan
        ]
    })

    return III, el_c_smooth, el_o_smooth



The function, which returns the info about average elasticities of characterustcs of 2 types of avocados (III), and also returns the 2 arrays of smoothed short-term (4-weeks) elasticities⬆️

In [1]:
reg = df[['year', 'Date', 'region', 'type', 'AveragePrice', 'Total Volume']].copy().sort_values(by=['type', 'region', 'year', 'Date'])
reg[['log_price', 'log_volume']] = np.log(reg[['AveragePrice', 'Total Volume']])
est_elasticity, est_reg_elasticity_c, est_reg_elasticity_o = regional_short_elasticity('TotalUS')

III = reg[['Date', 'region', 'type']].copy()
III['STM_elasticity'] = np.nan
III.loc[(III['region'] == 'TotalUS') & (III['type'] == 'conventional'), 'STM_elasticity'] = est_reg_elasticity_c
III.loc[(III['region'] == 'TotalUS') & (III['type'] == 'organic'), 'STM_elasticity'] = est_reg_elasticity_o

for i in reg['region'].unique():

  if i == 'TotalUS':
    continue

  reagional_STM_elasticity, est_reg_elasticity_c, est_reg_elasticity_o = regional_short_elasticity(i)

  est_elasticity = pd.concat([est_elasticity, reagional_STM_elasticity])

  III.loc[(III['region'] == i) & (III['type'] == 'conventional'), 'STM_elasticity'] = est_reg_elasticity_c
  III.loc[(III['region'] == i) & (III['type'] == 'organic'), 'STM_elasticity'] = est_reg_elasticity_o

#display(est_elasticity)
#print(III.head(40))

NameError: name 'df' is not defined

The for() block, which builds the final array for the step 3 - Smoothed(4w) short-term elasticities for each region for each week⬆️

## Task 4 (S)

Calculate long-term elasticity using 52-week rolling regression of log(volume) on log(price). This requires calculating slope coefficient over a rolling window, which can be approximated with covariance/variance formulas.

In [ ]:
reg = df[['year', 'Date', 'region', 'type', 'AveragePrice', 'Total Volume']].copy().sort_values(by=['type', 'region', 'year', 'Date'])
reg[['log_price', 'log_volume']] = np.log(reg[['AveragePrice', 'Total Volume']])

def regional_elasticity(i):

  c_mask = (reg['region'] == i) & (reg['type'] == 'conventional')
  o_mask = (reg['region'] == i) & (reg['type'] == 'organic')

  var_c = reg[c_mask]['log_price'].rolling(52).var().values
  var_o = reg[o_mask]['log_price'].rolling(52).var().values

  cov_c = reg[c_mask]['log_volume'].rolling(52).cov(reg[c_mask]['log_price']).values
  cov_o = reg[o_mask]['log_volume'].rolling(52).cov(reg[o_mask]['log_price']).values

  est_elasticity_c = cov_c / var_c
  est_elasticity_o = cov_o / var_o

  t4 = pd.DataFrame({'region' : [i, i], 'type' : ['conventional', 'organic'], 'elasticity' : [round(np.nanmean(est_elasticity_c), 3), round(np.nanmean(est_elasticity_o), 3)]})

  return t4, est_elasticity_c, est_elasticity_o

Dividing by the type and make the rolling window (size 52 (weeks)) which also automatially sets NaNs for the first 51 weeks, which is absolutely correct. The function also returns raw elasticity data on both types for the variable using in task 7.

In [ ]:
est_elasticity, est_reg_elasticity_c, est_reg_elasticity_o = regional_elasticity('TotalUS')

t7 = reg[['Date', 'region', 'type']].copy()
t7['LTM_elasticity'] = np.nan
t7.loc[(t7['region'] == 'TotalUS') & (t7['type'] == 'conventional'), 'LTM_elasticity'] = est_reg_elasticity_c
t7.loc[(t7['region'] == 'TotalUS') & (t7['type'] == 'organic'), 'LTM_elasticity'] = est_reg_elasticity_o

for i in reg['region'].unique():

  if i == 'TotalUS':
    continue

  reagional_LTM_elasticity, est_reg_elasticity_c, est_reg_elasticity_o = regional_elasticity(i)

  est_elasticity = pd.concat([est_elasticity, reagional_LTM_elasticity])

  t7.loc[(t7['region'] == i) & (t7['type'] == 'conventional'), 'LTM_elasticity'] = est_reg_elasticity_c
  t7.loc[(t7['region'] == i) & (t7['type'] == 'organic'), 'LTM_elasticity'] = est_reg_elasticity_o

display(est_elasticity)
print('Elasticity of volume on price given is the mean value of such an LTM indicator over the period of observation (2015-2018 inclusive).')

,region,type,elasticity
0,TotalUS,conventional,-0.889
1,TotalUS,organic,-0.896
0,Albany,conventional,-0.611
1,Albany,organic,-0.808
0,Atlanta,conventional,-1.118
...,...,...,...
1,Tampa,organic,-1.207
0,West,conventional,-1.041
1,West,organic,-1.315
0,WestTexNewMexico,conventional,-1.346


Elasticity of volume on price given is the mean value of such an LTM indicator over the period of observation (2015-2018 inclusive).


Program runs through all the regions (US total is on top of them as a highlight), using the function in the first sell to calculate the metric for the region and concat it to the current table. It also creating t7 variable as a dataset with all the elasticities needed for the final metrics in the 7th task.

## Task 5 (R)

For regions with both organic and conventional data calculate price premium (organic/conventional - 1), cross-price elasticity (organic demand change when conventional price changes), and market share trends.

Price Premium per Week

In [ ]:
conn = psycopg2.connect(
    host='localhost', database=DB_NAME,
    user=usr_nm, password=pwd, port=port
)
cur = conn.cursor()

cur.execute("""
    SELECT
    -- Here we take region and week from the table that will contain only conventional type of avocado
        c."region",
        c."Date",
        ROUND(c."AveragePrice", 4)                                    AS conv_price,
        ROUND(o."AveragePrice", 4)                                    AS org_price,
        ROUND(
            (o."AveragePrice" / c."AveragePrice" - 1) * 100,
        2)                                                            AS price_premium_pct
    FROM avocado_data c
    -- self-join: pair conventional and organic rows
    -- that share the same region AND the same week date
    --if there is no data in a region on organic avocado type it won't be in the table
    INNER JOIN avocado_data o
        ON  c."region" = o."region"
        AND c."Date"   = o."Date"
    -- pin each alias to its type
    WHERE c."type" = 'conventional'
      AND o."type" = 'organic'
    -- group by date — one row per region per week
    GROUP BY c."region", c."Date", c."AveragePrice", o."AveragePrice"
    -- sort chronologically within each region
    ORDER BY c."region", c."Date";
""")

data    = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_premium = pd.DataFrame(data=data, columns=columns)
print(df_premium.to_string(index=False))

cur.close()
conn.close()


What price premium means Price premium (%) = (P_organic / P_conventional − 1) × 100. A value of 20 means organic avocados cost 20% more than conventional in that region and week.

- Self-JOIN on `"region"` + `"Date"` pairs conventional and organic rows for the exact same date in the same region; INNER JOIN excludes regions lacking one type
- `WHERE c."type" = 'conventional' AND o."type" = 'organic'` pins each alias to exactly one avocado type because we have two:organic,conventional
- `GROUP BY "region", "Date"` we decided that weeks are the most useful

Cross-Price Elasticity of Organic Demand

In [ ]:
conn = psycopg2.connect(
    host='localhost', database=DB_NAME,
    user=usr_nm, password=pwd, port=port
)
cur = conn.cursor()

cur.execute("""
    SELECT
        "region",
        "year",
        ROUND(avg_conv_price, 4)                                      AS conv_price_curr,
        ROUND(prev_conv_price, 4)                                      AS conv_price_prev,

        -- how much did conventional price change vs last year? (in %)
        ROUND(
            (avg_conv_price - prev_conv_price)
            / prev_conv_price * 100, 2)                                AS conv_price_chg_pct,

        ROUND(avg_org_vol, 2)                                          AS org_vol_curr,
        ROUND(prev_org_vol, 2)                                         AS org_vol_prev,

        -- same thing but for organic volume
        ROUND(
            (avg_org_vol - prev_org_vol)
            / prev_org_vol * 100, 2)                                   AS org_vol_chg_pct,

        -- the big formula: % change in organic volume / % change in conv price
        -- if result > 0 — they're substitutes (organic demand follows conv price up)
        -- if result < 0 — they're complements (rare for avocados)
        ROUND(
            ((avg_org_vol    - prev_org_vol)    / prev_org_vol)
            /
            ((avg_conv_price - prev_conv_price) / prev_conv_price),
        4)                                                             AS cross_price_elasticity

    -- grab everything from the subquery below that already has LAG columns in it
    FROM (

        SELECT
            "region",
            "year",
            avg_conv_price,
            avg_org_vol,

            -- LAG just looks one row back within each region's window
            -- PARTITION BY "region" means Albany only looks at Albany rows,
            -- Atlanta only looks at Atlanta rows, etc.
            -- ORDER BY "year" makes sure "one row back" = one year back
            -- for the very first year (2015) there's no previous row → returns NULL
            -- we'll filter those NULLs out below
            LAG(avg_conv_price) OVER (PARTITION BY "region" ORDER BY "year") AS prev_conv_price,
            LAG(avg_org_vol)    OVER (PARTITION BY "region" ORDER BY "year") AS prev_org_vol

        -- this inner subquery is the same self-join we used for price premium:
        -- c = conventional, o = organic, joined by region + date
        -- GROUP BY collapses all 52 weeks into one row per region per year
        FROM (
            SELECT
                c."region",
                c."year",
                AVG(c."AveragePrice") AS avg_conv_price,
                AVG(o."Total Volume") AS avg_org_vol
            FROM avocado_data c
            INNER JOIN avocado_data o
                ON  c."region" = o."region"
                AND c."Date"   = o."Date"
            WHERE c."type" = 'conventional'
              AND o."type" = 'organic'
            GROUP BY c."region", c."year"
        ) AS yearly

    ) AS with_lag

    -- drop 2015 rows — LAG has nothing to look back at, so it returned NULL
    WHERE prev_conv_price IS NOT NULL

    -- also drop rows where price didn't change — we'd be dividing by zero, not fun
      AND prev_conv_price != avg_conv_price

    ORDER BY "region", "year";
""")

data    = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_elast = pd.DataFrame(data=data, columns=columns)
print(df_elast.to_string(index=False))

cur.close()
conn.close()

- CPE = (% Δ Q_organic) / (% Δ P_conventional). CPE > 0 means the two types are substitutes; consumers switch to organic when conventional prices rise.
If conventional avocados get 10% more expensive and organic sales go up 15% — that's a cross-price elasticity of 1.5. It tells us people are switching to organic when conventional gets pricey. They're substitutes.

- The innermost subquery yearly is the same self-join we already used in calculation of Price Premium for conventional, o for organic, joined by region + date, collapsed into one row per region per year with `GROUP BY`.

- The middle subquery with_lag takes those yearly averages and adds two extra columns using `LAG`. `LAG` is a window function — instead of collapsing rows like `GROUP BY` does, it just peeks at the previous row and copies its value into a new column. `PARTITION BY "region"` makes sure each region has its own separate window (Albany doesn't accidentally look at Atlanta's previous year). `ORDER BY "year"` inside the window tells `LAG` what "previous" actually means — one year back. For 2015 there's no previous year, so `LAG` returns NULL.



Organic Market Share

In [ ]:
# connect to the database, nothing new here
conn = psycopg2.connect(
    host='localhost', database=DB_NAME,
    user=usr_nm, password=pwd, port=port
)
cur = conn.cursor()

cur.execute("""
    SELECT
        o."region",
        o."year",
        ROUND(o.org_vol,  2)                                      AS organic_volume,
        ROUND(c.conv_vol, 2)                                      AS conventional_volume,

        -- just adding both volumes together to get the full market size
        ROUND(o.org_vol + c.conv_vol, 2)                          AS total_volume,

        -- the main metric: what % of total sales is organic?
        -- e.g. 100k organic out of 1M total = 10%
        -- if this number grows year over year -> organic is taking over
        ROUND(o.org_vol / (o.org_vol + c.conv_vol) * 100, 2)     AS organic_share_pct

    -- subquery o: total organic volume per region per year
    -- SUM instead of AVG because we want the actual total units sold,
    -- not the average weekly amount
    FROM (
        SELECT "region", "year", SUM("Total Volume") AS org_vol
        FROM   avocado_data
        WHERE  "type" = 'organic'
        GROUP BY "region", "year"
    ) AS o

    -- subquery c: same thing but for conventional
    INNER JOIN (
        SELECT "region", "year", SUM("Total Volume") AS conv_vol
        FROM   avocado_data
        WHERE  "type" = 'conventional'
        GROUP BY "region", "year"
    ) AS c

        -- glue them together: only keep region-year combos
        -- where BOTH organic and conventional data exist
        -- if a region only has one type it just won't show up
        ON  o."region" = c."region"
        AND o."year"   = c."year"

    -- sort by region then year so the trend is easy to read left to right
    ORDER BY o."region", o."year";
""")

# grab results from the server
data    = cur.fetchall()

# pull column names out of the cursor metadata
columns = [desc[0] for desc in cur.description]

# wrap everything in a DataFrame and print without the default row numbers
df_share = pd.DataFrame(data=data, columns=columns)
print(df_share.to_string(index=False))

cur.close()
conn.close()


- Organic market share is just: how many avocados sold were organic, out of the total. If that number grows year over year — people are genuinely shifting toward organic, not just a one-time spike.

- We split the query into two separate subqueries before joining them — one that only looks at organic rows, one that only looks at conventional. This way each subquery is clean and focused on one thing, instead of having a messy single query that tries to do both at once and needs extra filtering to stay correct.

- We use `SUM` instead of `AVG` here because we care about the total market size — how many avocados actually changed hands that year, not the average week. Market share math only makes sense on totals.

- The `INNER JOIN` on `(region, year)` does a nice thing — if a region only has organic data but no conventional (or vice versa) that year, it just quietly disappears from the results.

## Task 6 (R)

Identify leading regions (e.g., California, New York) and calculate price transmission: how price changes in major regions affect prices in smaller regions. Use lagged correlation analysis with 1-4 week lags.

Here we made the decision that it is not practical to consider aggregating regions — for example, if California turns out to be the sales leader, it would be rather odd to speak of a correlation between California's sales and those of Los Angeles, San Francisco, San Diego, and Sacramento, since these cities are part of California. Therefore, we excluded aggregating regions from the analysis.

In [ ]:
conn = psycopg2.connect(
    host='localhost', database=DB_NAME,
    user=usr_nm, password=pwd, port=port
)
cur = conn.cursor()

cur.execute("""
    SELECT
        "region",
        ROUND(SUM("Total Volume"), 0) AS total_volume
    FROM avocado_data
    WHERE "region" NOT IN (
        -- these contain cities that exist separately in the dataset,
        -- so their volume is just a sum of those cities — not a real market
        'TotalUS',
        'West',
        'California',
        'SouthCentral',
        'GreatLakes',
        'Northeast',
        'Southeast',
        'Midsouth',
        'Plains',
        'NorthernNewEngland'
        -- SouthCarolina and WestTexNewMexico are kept —
        -- they have no sub-cities in the dataset so they're real standalone markets
    )
    GROUP BY "region"
    ORDER BY total_volume DESC
    LIMIT 5;
""")

data       = cur.fetchall()
columns    = [desc[0] for desc in cur.description]
df_leaders = pd.DataFrame(data=data, columns=columns)
print(df_leaders)

cur.close()
conn.close()

- A "price leader" is a region with enough market depth that its price signals are reliable and timely. High-volume regions (US, California) trade enough units every week that their prices reflect fundamental supply/demand quickly, whereas thin markets react with a delay.
- We use `GROUP BY` and `ORDER BY total_volume` to search for regions with with the biggest "Total Volume"

In [ ]:
conn = psycopg2.connect(
    host='localhost', database=DB_NAME,
    user=usr_nm, password=pwd, port=port
)
cur = conn.cursor()
#leaders from the previous step
LEADER_1 = 'LosAngeles'
LEADER_2 = 'NewYork'

# aggregate regions excluded from the TARGET side too —
# same logic: their volume is just a sum of real cities,
# so correlation with them would be meaningless
AGGREGATES = (
    'TotalUS', 'West', 'California',
    'SouthCentral', 'GreatLakes', 'Northeast',
    'Southeast', 'Midsouth', 'Plains', 'NorthernNewEngland'
    # SouthCarolina and WestTexNewMexico stay in — they're standalone markets
)

cur.execute("""
    SELECT
        leading_region,
        target_region,
        -- all 4 lags in one row — easy to spot where correlation peaks
        ROUND(CORR(price_lag1, target_price)::NUMERIC, 4) AS corr_lag_1w,
        ROUND(CORR(price_lag2, target_price)::NUMERIC, 4) AS corr_lag_2w,
        ROUND(CORR(price_lag3, target_price)::NUMERIC, 4) AS corr_lag_3w,
        ROUND(CORR(price_lag4, target_price)::NUMERIC, 4) AS corr_lag_4w

    FROM (
        SELECT
            l."region" AS leading_region,
            t."region" AS target_region,

            -- LAG(price, N) looks N rows back within each region's window
            -- since data is weekly, N rows back = N weeks back
            -- price_lag1 = leader's price 1 week ago on date D,
            -- paired with target's current price on date D
            l.price_lag1,
            l.price_lag2,
            l.price_lag3,
            l.price_lag4,

            t.price AS target_price

        -- leader subquery: weekly avg price + 4 LAG columns
        FROM (
            SELECT
                "region",
                "Date",
                LAG(price, 1) OVER (PARTITION BY "region" ORDER BY "Date") AS price_lag1,
                LAG(price, 2) OVER (PARTITION BY "region" ORDER BY "Date") AS price_lag2,
                LAG(price, 3) OVER (PARTITION BY "region" ORDER BY "Date") AS price_lag3,
                LAG(price, 4) OVER (PARTITION BY "region" ORDER BY "Date") AS price_lag4
            FROM (
                -- compute weekly avg price first, THEN apply LAG on top —
                -- window functions can't sit directly on top of GROUP BY
                SELECT "region", "Date", AVG("AveragePrice") AS price
                FROM avocado_data
                GROUP BY "region", "Date"
            ) AS weekly_lead
        ) AS l

        -- target subquery: just the current week's price, no LAG needed
        INNER JOIN (
            SELECT "region", "Date", AVG("AveragePrice") AS price
            FROM avocado_data
            GROUP BY "region", "Date"
        ) AS t
            -- join on the SAME date D:
            -- leader carries prices from D-1w..D-4w via LAG,
            -- target has its current price on D —
            -- that's exactly what "N-week price transmission lag" means
            ON  l."Date"   = t."Date"
            AND t."region" NOT IN %s    -- exclude aggregate regions from targets
            AND t."region" NOT IN %s    -- exclude the leaders themselves

        WHERE l."region" IN %s          -- only look at our two leaders

    ) AS joined

    GROUP BY leading_region, target_region

    -- drop pairs where we didn't have enough data to compute correlation
    HAVING CORR(price_lag1, target_price) IS NOT NULL

    ORDER BY leading_region, target_region;
""",
(AGGREGATES,
 (LEADER_1, LEADER_2),
 (LEADER_1, LEADER_2))
)

data    = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_corr = pd.DataFrame(data=data, columns=columns)
print(df_corr.to_string(index=False))

cur.close()
conn.close()


- Price transmission: if LosAngeles prices jump this week, do prices in Albany follow a week later? The lag where correlation peaks is the answer — that's how fast a price signal travels from a big market to a smaller one.

- The key tool here is `LAG(price, N) OVER (PARTITION BY "region" ORDER BY "Date")` — it just looks N rows back within each region's sorted history. Since the dataset is weekly, N rows back = N weeks back. We compute all four lags in one pass and get four correlation columns side by side.

- We can't apply `LAG` directly on top of `GROUP BY` in the same `SELECT`, so we nest it: the inner subquery computes weekly average prices first, the outer one applies `LAG` on top of those averages. Two layers, but each does one simple thing.

## Task 7 (I)

Create final analytical table with one row per region per week containing: original price and volume, normalized metrics, short and long-term elasticities, organic premium, inventory cycle phase, seasonal factors, and next-week price forecast using multiple models.

In [ ]:
wide = reg.pivot_table(
    index=['region', 'Date'],
    columns='type',
    values=['AveragePrice', 'Total Volume'],
    aggfunc='first'
).round(3)

wide.columns = ['_'.join(col).strip() for col in wide.columns]
wide = wide.reset_index()
wide = wide.merge(III[['region','Date','STM_elasticity']], on=['region', 'Date'], how='right')
wide['LTM_elasticity'] = t7['LTM_elasticity'].values


In [ ]:
wide['Organic premium']=df_premium['price_premium_pct'] #SQL stuff

In [ ]:
def get_inventory_phase(vol_series, window=12):
    roll_mean = vol_series.rolling(window=window, min_periods=1).mean()
    ratio = vol_series / roll_mean
    return pd.cut(ratio,
                  bins=[-np.inf, 0.8, 1.2, np.inf],
                  labels=['shortage', 'normal', 'surplus']).astype(str)

wide['inventory_phase_conv'] = wide.groupby('region')['Total Volume_conventional'].transform(get_inventory_phase)
wide['inventory_phase_org']  = wide.groupby('region')['Total Volume_organic'].transform(get_inventory_phase)

In [ ]:
final = pd.DataFrame()
final=wide

final['price_mean_conv'] = final.groupby(['region'])['AveragePrice_conventional'].transform('mean')
final['price_std_conv'] = final.groupby(['region'])['AveragePrice_conventional'].transform('std')
final['volume_mean_conv'] = final.groupby(['region'])['Total Volume_conventional'].transform('mean')
final['volume_std_conv'] = final.groupby(['region'])['Total Volume_conventional'].transform('std')

final['price_mean_org'] = final.groupby(['region'])['AveragePrice_organic'].transform('mean')
final['price_std_org'] = final.groupby(['region'])['AveragePrice_organic'].transform('std')
final['volume_mean_org'] = final.groupby(['region'])['Total Volume_organic'].transform('mean')
final['volume_std_org'] = final.groupby(['region'])['Total Volume_organic'].transform('std')

final['price_z_conv(norm)'] = (final['AveragePrice_conventional'] - final['price_mean_conv']) / final['price_std_conv']
final['volume_z_conv(norm)'] = (final['Total Volume_conventional'] - final['volume_mean_conv']) / final['volume_std_conv']

final['price_z_org(norm)'] = (final['AveragePrice_organic'] - final['price_mean_org']) / final['price_std_org']
final['volume_z_org(norm)'] = (final['Total Volume_organic'] - final['volume_mean_org']) / final['volume_std_org']

final = final.drop(columns=[
    'price_mean_conv', 'price_std_conv', 'volume_mean_conv', 'volume_std_conv',
    'price_mean_org', 'price_std_org', 'volume_mean_org', 'volume_std_org'
])

In [ ]:
final['Date'] = pd.to_datetime(final['Date'])

final['week_of_year'] = final['Date'].dt.isocalendar().week

final['avg_price_by_week_conv'] = final.groupby(['region', 'week_of_year'])['AveragePrice_conventional'].transform('mean')
final['avg_price_by_week_org'] = final.groupby(['region', 'week_of_year'])['AveragePrice_organic'].transform('mean')

final['seasonal_factor_conv'] = final['AveragePrice_conventional'] / final['avg_price_by_week_conv']
final['seasonal_factor_org'] = final['AveragePrice_organic'] / final['avg_price_by_week_org']

In [ ]:
grp = final.groupby(['region'])
final['forecast_naive_conv'] = grp['AveragePrice_conventional'].transform(lambda x: x.shift(1))
final['forecast_naive_org'] = grp['AveragePrice_organic'].transform(lambda x: x.shift(1))
final['forecast_ma4_conv']   = grp['AveragePrice_conventional'].transform(lambda x: x.rolling(4, min_periods=1).mean())
final['forecast_ma4_org']   = grp['AveragePrice_organic'].transform(lambda x: x.rolling(4, min_periods=1).mean())

p_lag1_conv = grp['AveragePrice_conventional'].transform(lambda x: x.shift(1))
p_lag4_conv = grp['AveragePrice_conventional'].transform(lambda x: x.shift(4))
p_lag1_org = grp['AveragePrice_organic'].transform(lambda x: x.shift(1))
p_lag4_org = grp['AveragePrice_organic'].transform(lambda x: x.shift(4))

final['forecast_trend_conv']    = (p_lag1_conv + (p_lag1_conv - p_lag4_conv) / 3.0).round(4)
final['forecast_trend_org']    = (p_lag1_org + (p_lag1_org - p_lag4_org) / 3.0).round(4)

final['forecast_ensemble_conv'] = final[['forecast_naive_conv','forecast_ma4_conv','forecast_trend_conv']].mean(axis=1).round(4)
final['forecast_ensemble_org'] = final[['forecast_naive_org','forecast_ma4_org','forecast_trend_org']].mean(axis=1).round(4)

final = final.drop(columns=[
    'forecast_naive_conv','forecast_ma4_conv','forecast_trend_conv',
    'forecast_naive_org','forecast_ma4_org','forecast_trend_org'
])